# Dashcam Change Analysis Validation

This notebook validates `dashcam_change_analysis.py` step by step on a dashcam video. It checks frame loading, pixel-change scoring, YOLO detections, and object entry/exit inference.

In [ ]:
import os

# Temporary workaround for OpenMP runtime collisions between packages like numpy and opencv.
# Prefer a clean conda environment with consistent package sources when possible.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print('Set KMP_DUPLICATE_LIB_OK=TRUE before importing OpenCV / NumPy / Ultralytics')

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from dashcam_change_analysis import (
    AnalysisConfig,
    analyze_top_changed_frames,
    build_model,
    compute_frame_change_score,
    detect_object_transitions,
    draw_annotations,
    extract_frames,
    load_video,
    run_yolo_detection,
    save_analysis_results,
    select_top_changed_frames,
    summarize_result,
)


In [ ]:
# Edit only these two values for your own data.
video_path = Path('12Feb2022/VID_004.MOV')
model_name_or_path = 'yolov8n.pt'  # Or a local Ultralytics weights file, for example Path('weights/custom.pt')

# Analysis controls. Adjust these if you want to tune the pipeline.
sample_interval = 5
top_n = 5
diff_threshold = 25
confidence_threshold = 0.25
iou_threshold = 0.3
resize_width = 960

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:

def show_image(image, title='', figsize=(12, 7), cmap=None):
    fig, ax = plt.subplots(figsize=figsize)
    if image.ndim == 2:
        ax.imshow(image, cmap=cmap or 'gray')
    else:
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
    plt.show()

def show_frame_grid(images, titles=None, cols=2, figsize=(16, 10)):
    if not images:
        print('No images to display.')
        return
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for index, image in enumerate(images):
        ax = axes[index]
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        if titles and index < len(titles):
            ax.set_title(titles[index])
        ax.axis('off')
    for index in range(len(images), len(axes)):
        axes[index].axis('off')
    plt.tight_layout()
    plt.show()

def detections_table(detections):
    return pd.DataFrame(detections)

## Video Inspection

This step proves that the video opens correctly and that sampled frames look reasonable before any analysis is run.

In [ ]:
capture, metadata = load_video(video_path)
print(f'Video: {video_path}')
print(f'Frame count: {metadata.frame_count}')
print(f'FPS: {metadata.fps:.2f}')
print(f'Width x Height: {metadata.width} x {metadata.height}')
capture.release()


In [ ]:
samples, metadata = extract_frames(
    video_path,
    interval=sample_interval,
    resize_width=resize_width,
    max_frames=metadata.frame_count,
)
print(f'Sampled frames: {len(samples)}')
print('First few sampled frames:')
for sample in samples[:6]:
    print(f'  frame={sample.frame_index:6d}  time={sample.timestamp_seconds:8.2f}s  shape={sample.frame.shape}')

show_frame_grid(
    [sample.frame for sample in samples[:6]],
    [f'Frame {sample.frame_index} @ {sample.timestamp_seconds:.1f}s' for sample in samples[:6]],
    cols=3,
    figsize=(18, 10),
)

## Frame-Change Scoring

This step computes a pixel-level change score for each sampled frame pair and selects the most changed events.

In [ ]:
all_scores = []
for previous_sample, current_sample in zip(samples[:-1], samples[1:]):
    change_score, changed_pixels, changed_ratio, diff_mask, abs_diff = compute_frame_change_score(
        previous_sample.frame,
        current_sample.frame,
        diff_threshold=diff_threshold,
    )
    all_scores.append(
        {
            'previous_frame_index': previous_sample.frame_index,
            'frame_index': current_sample.frame_index,
            'timestamp_seconds': current_sample.timestamp_seconds,
            'change_score': change_score,
            'changed_pixels': changed_pixels,
            'changed_ratio': changed_ratio,
            'diff_mask': diff_mask,
            'abs_diff': abs_diff,
        }
    )

scores_df = pd.DataFrame(all_scores)

print("size of scores_df:", scores_df.shape[0])
display(scores_df[['previous_frame_index', 'frame_index', 'timestamp_seconds', 'change_score', 'changed_pixels', 'changed_ratio']].head(10))


In [ ]:
for index, row in scores_df['timestamp_seconds'].items():
    print(f"Index: {index}, Timestamp: {row}")

In [ ]:

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(scores_df['frame_index'], scores_df['change_score'], marker='.', linewidth=1.5)
ax.set_title('Pixel-change score by sampled frame')
ax.set_xlabel('Frame index')
ax.set_ylabel('Change score')
# start x axis at 0
ax.set_xlim(left=0)


# add second x axis with timestamp in seconds
x2 = ax.twiny()
x2.set_xlim(min(scores_df['timestamp_seconds']), max(scores_df['timestamp_seconds']))
# change color of the second x axis to green
x2.spines['top'].set_color('green')
x2.set_xlabel('Timestamp (seconds)')
plt.show()


In [ ]:
top_changed = select_top_changed_frames(
    samples,
    top_n=top_n,
    diff_threshold=diff_threshold,
)
top_changed_df = pd.DataFrame([
    {
        'frame_index': item['frame_index'],
        'previous_frame_index': item['previous_frame_index'],
        'timestamp_seconds': item['timestamp_seconds'],
        'change_score': item['change_score'],
        'changed_pixels': item['changed_pixels'],
        'changed_ratio': item['changed_ratio'],
    }
    for item in top_changed
])


### graph with frames changed

In [ ]:

display(top_changed_df)

## Visual Verification

This step shows the top changed frames and their raw difference masks so the score can be checked visually.

In [ ]:
show_frame_grid(
    [item['frame'] for item in top_changed],
    [f"Frame {item['frame_index']} | score={item['change_score']:.0f}" for item in top_changed],
    cols=2,
    figsize=(18, 10),
)

if top_changed:
    first_event = top_changed[0]
    show_image(first_event['diff_mask'], 
               title=f"Difference mask for frame {first_event['frame_index']}", 
               figsize=(12, 6), cmap='coolwarm')
    show_image(first_event['abs_diff'], 
               title=f"Absolute difference for frame {first_event['frame_index']}", 
               figsize=(12, 6), cmap='coolwarm')

## YOLO Detection Validation

This step runs Ultralytics YOLO on the most changed frames and prints the detections in a table.

In [ ]:
model = build_model(model_name_or_path)
detection_runs = []

for item in top_changed:
    detections = run_yolo_detection(
        model=model,
        frame=item['frame'],
        confidence_threshold=confidence_threshold,
        object_limit=25,
    )
    detection_runs.append((item, detections))
    print(f"Frame {item['frame_index']} | detections={len(detections)} | score={item['change_score']:.0f}")
    if detections:
        display(detections_table([
            {
                'class_name': detection.class_name,
                'class_id': detection.class_id,
                'confidence': detection.confidence,
                'x1': detection.bbox[0],
                'y1': detection.bbox[1],
                'x2': detection.bbox[2],
                'y2': detection.bbox[3],
            }
            for detection in detections
        ]))
    else:
        print('  No detections above threshold.')

    annotated = draw_annotations(
        frame=item['frame'],
        detections=detections,
        diff_mask=item['diff_mask'],
    )
    show_image(annotated, title=f"YOLO detections on frame {item['frame_index']}", figsize=(14, 8))

## Entry/Exit Analysis

This step compares adjacent highly changed frames and shows which objects entered, exited, or persisted.

In [ ]:
sample_by_index = {sample.frame_index: sample for sample in samples}

for item in top_changed[:min(3, len(top_changed))]:
    previous_sample = sample_by_index[item['previous_frame_index']]
    current_sample = sample_by_index[item['frame_index']]
    previous_detections = run_yolo_detection(
        model=model,
        frame=previous_sample.frame,
        confidence_threshold=confidence_threshold,
        object_limit=25,
    )
    current_detections = run_yolo_detection(
        model=model,
        frame=current_sample.frame,
        confidence_threshold=confidence_threshold,
        object_limit=25,
    )
    entered_objects, exited_objects, matches = detect_object_transitions(
        previous_detections=previous_detections,
        current_detections=current_detections,
        iou_threshold=iou_threshold,
    )

    print(f"Frame {current_sample.frame_index} compared with {previous_sample.frame_index}")
    print(f"  entered: {len(entered_objects)} | exited: {len(exited_objects)} | matched: {len(matches)}")

    if entered_objects:
        print('  Entered objects:')
        display(detections_table([
            {
                'class_name': detection.class_name,
                'confidence': detection.confidence,
                'bbox': detection.bbox,
            }
            for detection in entered_objects
        ]))

    if exited_objects:
        print('  Exited objects:')
        display(detections_table([
            {
                'class_name': detection.class_name,
                'confidence': detection.confidence,
                'bbox': detection.bbox,
            }
            for detection in exited_objects
        ]))

    if matches:
        print('  Matched objects:')
        display(pd.DataFrame([
            {
                'previous_class': match.previous.class_name,
                'current_class': match.current.class_name,
                'iou': match.iou,
                'previous_confidence': match.previous.confidence,
                'current_confidence': match.current.confidence,
            }
            for match in matches
        ]))

    annotated = draw_annotations(
        frame=current_sample.frame,
        detections=current_detections,
        entered_objects=entered_objects,
        exited_objects=exited_objects,
        matches=matches,
        diff_mask=item['diff_mask'],
    )
    show_image(annotated, title=f"Transitions for frame {current_sample.frame_index}", figsize=(14, 8))

## End-to-End Test

This step runs the full pipeline on the same video and prints a compact summary for each selected change event.

In [ ]:
config = AnalysisConfig(
    resize_width=resize_width,
    diff_threshold=diff_threshold,
    top_n=top_n,
    sample_interval=sample_interval,
    iou_threshold=iou_threshold,
    confidence_threshold=confidence_threshold,
    object_limit=25,
)

summary = analyze_top_changed_frames(video_path, model_name_or_path, config)
print(f'End-to-end results: {len(summary.results)}')
for result in summary.results:
    compact = summarize_result(result)
    print(
        f"Frame {compact['frame_index']} | score={compact['change_score']:.0f} | "
        f"entered={len(compact['entered_objects'])} | exited={len(compact['exited_objects'])} | matched={len(compact['matched_objects'])}"
    )

# Optional: save annotated outputs for later review.
# saved_files = save_analysis_results(summary)
# print(saved_files)

## bar plot with identified objects

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(scores_df['frame_index'], scores_df['change_score'], marker='.', linewidth=1.5)
# include timestamp on the second x-axis

# add second x axis with timestamp in seconds
x2 = ax.twiny()
x2.set_xlim(min(scores_df['timestamp_seconds']), max(scores_df['timestamp_seconds']))
# change color of the second x axis to green
x2.spines['top'].set_color('green')
x2.set_xlabel('Timestamp (seconds)')

'''
# mark the top changed frames on the plot
for item in top_changed:
    ax.axvline(x=item['frame_index'], color='red', linestyle='--', alpha=0.5)
    ax.text(
        item['frame_index'],
        item['change_score'] + 5,
        f"{item['frame_index']}\n{item['timestamp_seconds']:.1f}s",
        color='red',
        fontsize=8,
        ha='center',
    )
'''
# use detection_runs to add class_name for the top changed frames
for item, detections in detection_runs:
    if detections:
        class_names = ', '.join([detection.class_name for detection in detections])
        ax.text(
            item['frame_index'],
            item['change_score'] + 15,
            f"Detected: {class_names}",
            color='blue',
            fontsize=8,
            ha='center',
        )

# label y axis
ax.set_ylabel('Change score')

# label x axis
ax.set_xlabel('Frame index')

# get y ticks from the scores_df
y_ticks = np.arange(0, scores_df['change_score'].max() + 20, 10)
ax.set_yticks(y_ticks+20)

# start x axis at 0
ax.set_xlim(left=0)

plt.show()

In [ ]:
# save a csv with the top changed frames and their detection results
results_list = []
for item, detections in detection_runs:
    results_list.append({
        'timestamp_seconds': round(item['timestamp_seconds'], 2),
        'frame_index': item['frame_index'],
        'previous_frame_index': item['previous_frame_index'],
        'change_score': round(item['change_score'], 2),
        'detections': ', '.join([detection.class_name for detection in detections]) if detections else '',
    })


## Save the results to a CSV file

In [ ]:

results_df = pd.DataFrame(results_list)
results_df.to_csv(f'./output/{video_path.stem}_top_changed_frames_detections.csv', index=False)